# 🟦 ViniVideo AI — Motor Kaggle

Este é o plano B do ViniVideo AI quando a GPU gratuita do Colab fica indisponível.

## Antes de rodar

1. Faça login no Kaggle.
2. Sua conta precisa estar verificada por telefone para liberar GPU.
3. Abra **Settings → Accelerator → GPU T4 x2**.
4. Deixe **Internet** ativada.
5. Use **Run All**.

Este notebook **não abre túnel público** e não conecta diretamente ao app. Ele gera o MP4 dentro do Kaggle e mostra um link para baixar.

O modo rápido usa **AnimateDiff-Lightning 4-step**.


In [ ]:
import os
import torch

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Ative Settings → Accelerator → GPU T4 x2 no Kaggle."
    )

print("GPU principal:", torch.cuda.get_device_name(0))
print("Quantidade de GPUs:", torch.cuda.device_count())


In [ ]:
!pip -q install --upgrade-strategy only-if-needed "diffusers==0.33.1" "transformers==4.51.3" "huggingface-hub==0.30.2" "tokenizers>=0.21,<0.22" "accelerate>=1.2,<2" "safetensors>=0.4" imageio imageio-ffmpeg hf-xet
print("Dependências prontas ✓")


In [ ]:
import gc
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from diffusers import (
    AnimateDiffPipeline,
    EulerDiscreteScheduler,
    MotionAdapter,
)

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MOTION_REPO = "ByteDance/AnimateDiff-Lightning"
MOTION_FILE = "animatediff_lightning_4step_diffusers.safetensors"

print("⚡ Carregando motor Rápido...")

device = "cuda"
dtype = torch.float16

adapter = MotionAdapter().to(device, dtype)
motion_path = hf_hub_download(
    repo_id=MOTION_REPO,
    filename=MOTION_FILE,
)
adapter.load_state_dict(
    load_file(motion_path, device=device),
    strict=True,
)

pipe = AnimateDiffPipeline.from_pretrained(
    BASE_MODEL,
    motion_adapter=adapter,
    torch_dtype=dtype,
    variant="fp16",
    use_safetensors=True,
    low_cpu_mem_usage=True,
    safety_checker=None,
)

pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config,
    timestep_spacing="trailing",
    beta_schedule="linear",
)

try:
    pipe.enable_vae_slicing()
except Exception:
    pass

pipe.enable_model_cpu_offload()

print("✅ Motor Rápido pronto")


In [ ]:
# ===== CONFIGURAÇÃO DO SEU VÍDEO =====
PROMPT = "Uma arara azul explorando uma cidade tropical, animação cinematográfica, movimento de câmera suave"
DURATION_SECONDS = 5
ASPECT = "9:16"       # "9:16", "16:9" ou "1:1"
SEED = 12345

# Dica: comece com 5 segundos. Depois teste 10, 15 ou 30.
print("Prompt:", PROMPT)
print("Duração:", DURATION_SECONDS, "s")
print("Formato:", ASPECT)


In [ ]:
from pathlib import Path
from diffusers.utils import export_to_video
from IPython.display import Video, display, FileLink
import subprocess
import torch
import gc

OUTPUT_DIR = Path("/kaggle/working/ViniVideoAI")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE = (
    "worst quality, low quality, blurry, jittery, distorted anatomy, "
    "deformed face, duplicated subject, flicker, text, watermark"
)

def aspect_size(aspect):
    if aspect == "9:16":
        return 288, 512
    if aspect == "1:1":
        return 384, 384
    return 512, 288

def make_scene_prompt(base_prompt, scene_index, total_scenes):
    if total_scenes <= 1:
        return base_prompt
    if scene_index == 0:
        beat = "establish the environment and main subject clearly"
    elif scene_index == total_scenes - 1:
        beat = "finish the action naturally with a cinematic ending"
    else:
        beat = "continue the same action and visual identity"
    return (
        base_prompt
        + ". "
        + beat
        + ". Keep subject design, palette and environment consistent."
    )

width, height = aspect_size(ASPECT)

scene_duration = 5
scene_count = max(
    1,
    (int(DURATION_SECONDS) + scene_duration - 1) // scene_duration,
)

scene_paths = []

print(f"🎬 Gerando {scene_count} tomada(s)...")

for scene_index in range(scene_count):
    start = scene_index * scene_duration
    remaining = int(DURATION_SECONDS) - start
    this_duration = max(1, min(scene_duration, remaining))

    scene_path = OUTPUT_DIR / f"scene_{scene_index + 1:03d}.mp4"

    if scene_path.exists() and scene_path.stat().st_size > 1024:
        print(f"♻️ Tomada {scene_index + 1} já existe; reutilizando checkpoint.")
        scene_paths.append(scene_path)
        continue

    frames_count = 32 if this_duration >= 4 else 24
    fps = max(5, min(10, round(frames_count / this_duration)))

    scene_prompt = make_scene_prompt(
        PROMPT,
        scene_index,
        scene_count,
    )

    print(
        f"⚡ Tomada {scene_index + 1}/{scene_count} "
        f"• {this_duration}s • {width}x{height}"
    )

    generator = torch.Generator(device="cpu").manual_seed(
        int(SEED) + scene_index
    )

    result = pipe(
        prompt=scene_prompt,
        negative_prompt=NEGATIVE,
        width=width,
        height=height,
        num_frames=frames_count,
        guidance_scale=1.0,
        num_inference_steps=4,
        generator=generator,
    )

    frames = result.frames[0]
    export_to_video(
        frames,
        str(scene_path),
        fps=fps,
    )
    scene_paths.append(scene_path)

    del result, frames
    gc.collect()
    torch.cuda.empty_cache()

    print(f"✅ Tomada {scene_index + 1} salva")

final_path = OUTPUT_DIR / "ViniVideoAI_Kaggle.mp4"

concat_file = OUTPUT_DIR / "concat.txt"
with open(concat_file, "w", encoding="utf-8") as f:
    for path in scene_paths:
        safe = str(path).replace("'", "'\\''")
        f.write(f"file '{safe}'\n")

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-f",
        "concat",
        "-safe",
        "0",
        "-i",
        str(concat_file),
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "18",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(final_path),
    ],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print()
print("✅ VÍDEO PRONTO:", final_path)
display(Video(str(final_path), embed=True))
display(FileLink(str(final_path), result_html_prefix="⬇️ Baixar MP4: "))


## Como usar no celular

Quando terminar, toque no link **⬇️ Baixar MP4** que aparece na última célula.

Se quiser gerar outro vídeo:
- altere apenas a célula **CONFIGURAÇÃO DO SEU VÍDEO**;
- mude o prompt/duração/formato;
- apague os arquivos de cena antigos em `/kaggle/working/ViniVideoAI` se quiser começar do zero;
- execute novamente a célula final.

O Kaggle tem limites e disponibilidade de GPU; este modo é um fallback, não uma GPU ilimitada.
